In [1]:
%pip install pandas numpy scikit-learn pymysql sqlalchemy cryptography

ERROR: Could not install packages due to an OSError: Could not find a suitable TLS CA certificate bundle, invalid path: C:\Program Files\PostgreSQL\18\ssl\certs\ca-bundle.crt



In [2]:
pip install pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import json

RAW_DATA_PATH = "data/raw_json"

files = [f for f in os.listdir(RAW_DATA_PATH) if f.endswith(".json")]
print(f"Total IPL match files found: {len(files)}")

sample_file_path = os.path.join(RAW_DATA_PATH, files[0])
with open(sample_file_path, "r") as f:
    sample_data = json.load(f)

print("\n--- Match Info Keys ---")
print(list(sample_data.get("info", {}).keys()))

print("\n--- Sample Teams & Venue ---")
print("Teams:", sample_data.get("info", {}).get("teams"))
print("Venue:", sample_data.get("info", {}).get("venue"))
print("Season:", sample_data.get("info", {}).get("season"))

Total IPL match files found: 0


IndexError: list index out of range

In [4]:
import os
print("Current Notebook Directory:", os.getcwd())
print("Contents of current directory:", os.listdir())

Current Notebook Directory: D:\ipl_analytics
Contents of current directory: ['.ipynb_checkpoints', 'data', 'Untitled.ipynb']


In [5]:
import os

print("Contents of 'data':", os.listdir("data"))

if "raw_json" in os.listdir("data"):
    print("Contents of 'data/raw_json':", os.listdir("data/raw_json")[:10])

Contents of 'data': ['raw_json']
Contents of 'data/raw_json': ['ipl_json']


In [7]:
import os
import json

RAW_DATA_PATH = "data/raw_json/ipl_json"

files = [f for f in os.listdir(RAW_DATA_PATH) if f.endswith(".json")]
print(f"Total IPL match files found: {len(files)}")

sample_file_path = os.path.join(RAW_DATA_PATH, files[0])
with open(sample_file_path, "r") as f:
    sample_data = json.load(f)

print("\n--- Match Info Keys ---")
print(list(sample_data.get("info", {}).keys()))

print("\n--- Sample Teams & Venue ---")
print("Teams:", sample_data.get("info", {}).get("teams"))
print("Venue:", sample_data.get("info", {}).get("venue"))
print("Season:", sample_data.get("info", {}).get("season"))

Total IPL match files found: 1243

--- Match Info Keys ---
['balls_per_over', 'city', 'dates', 'event', 'gender', 'match_type', 'officials', 'outcome', 'overs', 'player_of_match', 'players', 'registry', 'season', 'team_type', 'teams', 'toss', 'venue']

--- Sample Teams & Venue ---
Teams: ['Sunrisers Hyderabad', 'Royal Challengers Bangalore']
Venue: Rajiv Gandhi International Stadium, Uppal
Season: 2017


In [8]:
import os
import json
import pandas as pd
from tqdm import tqdm

RAW_DATA_PATH = "data/raw_json/ipl_json"
files = [f for f in os.listdir(RAW_DATA_PATH) if f.endswith(".json")]

matches_list = []
deliveries_list = []

print(f"Parsing {len(files)} matches...")

for filename in tqdm(files):
    match_id = filename.replace(".json", "")
    filepath = os.path.join(RAW_DATA_PATH, filename)
    
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    info = data.get("info", {})
    venue = info.get("venue", "Unknown")
    dates = info.get("dates", ["Unknown"])
    season = str(info.get("season", "Unknown"))
    teams = info.get("teams", ["Unknown", "Unknown"])
    toss = info.get("toss", {})
    outcome = info.get("outcome", {})
    
    matches_list.append({
        "match_id": match_id,
        "season": season,
        "date": dates[0] if dates else "Unknown",
        "venue": venue,
        "city": info.get("city", "Unknown"),
        "team1": teams[0] if len(teams) > 0 else "Unknown",
        "team2": teams[1] if len(teams) > 1 else "Unknown",
        "toss_winner": toss.get("winner", "Unknown"),
        "toss_decision": toss.get("decision", "Unknown"),
        "winner": outcome.get("winner", "No Result")
    })
    
    
    innings = data.get("innings", [])
    for inn_idx, inn in enumerate(innings):
        inn_num = inn_idx + 1
        batting_team = inn.get("team", "Unknown")
        
        
        batter_balls_faced = {}
        
        for over_data in inn.get("overs", []):
            over_num = over_data.get("over", 0)  
            
            if over_num < 6:
                phase = "Powerplay"
            elif over_num < 15:
                phase = "Middle"
            else:
                phase = "Death"
                
            for ball_idx, delivery in enumerate(over_data.get("deliveries", [])):
                batter = delivery.get("batter", "Unknown")
                bowler = delivery.get("bowler", "Unknown")
                non_striker = delivery.get("non_striker", "Unknown")
                runs = delivery.get("runs", {})
                extras = delivery.get("extras", {})
                
                is_wide = "wides" in extras
                if not is_wide:
                    batter_balls_faced[batter] = batter_balls_faced.get(batter, 0) + 1
                    
                balls_faced_so_far = batter_balls_faced.get(batter, 1)
                
                wickets = delivery.get("wickets", [])
                is_wicket = 1 if len(wickets) > 0 else 0
                dismissal_kind = wickets[0].get("kind") if is_wicket else "not out"
                player_out = wickets[0].get("player_out") if is_wicket else None
                
                deliveries_list.append({
                    "match_id": match_id,
                    "inning": inn_num,
                    "over_num": over_num + 1, 
                    "ball_num": ball_idx + 1,
                    "match_phase": phase,
                    "batting_team": batting_team,
                    "batter": batter,
                    "bowler": bowler,
                    "non_striker": non_striker,
                    "batter_balls_faced": balls_faced_so_far,
                    "latency_bucket": "Cold Start (Balls 1-8)" if balls_faced_so_far <= 8 else "Set (Balls 9+)",
                    "batsman_runs": runs.get("batter", 0),
                    "extra_runs": runs.get("extras", 0),
                    "total_runs": runs.get("total", 0),
                    "is_dot": 1 if runs.get("total", 0) == 0 else 0,
                    "is_boundary": 1 if runs.get("batter", 0) in [4, 6] else 0,
                    "is_wicket": is_wicket,
                    "dismissal_kind": dismissal_kind,
                    "player_out": player_out
                })

df_matches = pd.DataFrame(matches_list)
df_deliveries = pd.DataFrame(deliveries_list)

print(f"\nDone! Extracted {len(df_matches)} matches and {len(df_deliveries)} deliveries.")

C:\ProgramData\Anaconda3-2022\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\ProgramData\Anaconda3-2022\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


Parsing 1243 matches...


100%|█████████████████████████████████████████████████████████████████████████████| 1243/1243 [00:11<00:00, 104.68it/s]



Done! Extracted 1243 matches and 295732 deliveries.


In [9]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


df_all = df_deliveries.merge(df_matches[["match_id", "venue"]], on="match_id", how="left")


venue_profiles = df_all.groupby("venue").agg(
    total_balls=("ball_num", "count"),
    run_rate=("total_runs", lambda x: (x.sum() * 6.0) / len(x)),
    boundary_pct=("is_boundary", lambda x: (x.sum() / len(x)) * 100),
    dot_ball_pct=("is_dot", lambda x: (x.sum() / len(x)) * 100),
    wickets_per_100_balls=("is_wicket", lambda x: (x.sum() / len(x)) * 100)
).reset_index()


venue_profiles = venue_profiles[venue_profiles["total_balls"] >= 2400].copy()


features = ["run_rate", "boundary_pct", "dot_ball_pct", "wickets_per_100_balls"]
scaler = StandardScaler()
scaled_features = scaler.fit_transform(venue_profiles[features])


kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
venue_profiles["cluster_id"] = kmeans.fit_predict(scaled_features)

cluster_summary = venue_profiles.groupby("cluster_id")[features].mean()
print("Cluster Statistical Profiles:")
display(cluster_summary)

Cluster Statistical Profiles:


,run_rate,boundary_pct,dot_ball_pct,wickets_per_100_balls
cluster_id,,,,
0,7.821933,15.703153,34.798325,4.933273
1,8.877747,19.318556,31.438772,4.867711
2,7.424435,14.587997,38.014568,5.475753


In [10]:
import os

os.makedirs("data/processed", exist_ok=True)

df_matches.to_csv("data/processed/ipl_matches.csv", index=False)
df_deliveries.to_csv("data/processed/ipl_deliveries.csv", index=False)
venue_profiles.to_csv("data/processed/venue_archetypes.csv", index=False)

print("Saved CSVs to data/processed/:")
print("- ipl_matches.csv")
print("- ipl_deliveries.csv")
print("- venue_archetypes.csv")

Saved CSVs to data/processed/:
- ipl_matches.csv
- ipl_deliveries.csv
- venue_archetypes.csv


In [11]:
from sqlalchemy import create_engine

DB_USER = "root"
DB_PASS = "007007"  
DB_HOST = "localhost"
DB_PORT = "3306"
DB_NAME = "ipl_tactics_db"

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

print("Importing Matches...")
df_matches.to_sql("matches", con=engine, if_exists="append", index=False, chunksize=1000)

print("Importing Venue Archetypes...")
venue_profiles.to_sql("venue_archetypes", con=engine, if_exists="append", index=False)

print("Importing Deliveries (this will take ~30-45 seconds for ~250k rows)...")
df_deliveries.to_sql("deliveries", con=engine, if_exists="append", index=False, chunksize=5000)

print("Done! All records inserted successfully.")

ModuleNotFoundError: No module named 'pymysql'

In [12]:
%pip install pymysql cryptography

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: Could not find a suitable TLS CA certificate bundle, invalid path: C:\Program Files\PostgreSQL\18\ssl\certs\ca-bundle.crt



In [13]:
import os

for var in ["REQUESTS_CA_BUNDLE", "CURL_CA_BUNDLE", "SSL_CERT_FILE"]:
    if var in os.environ:
        del os.environ[var]

!pip install --trusted-host pypi.org --trusted-host files.pythonhosted.org pymysql cryptography

In [14]:
from sqlalchemy import create_engine

DB_USER = "root"
DB_PASS = "{PASSWORD}"  
DB_HOST = "localhost"
DB_PORT = "{PORT}"
DB_NAME = "ipl_tactics_db"

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

print("Importing Matches...")
df_matches.to_sql("matches", con=engine, if_exists="append", index=False, chunksize=1000)

print("Importing Venue Archetypes...")
venue_profiles.to_sql("venue_archetypes", con=engine, if_exists="append", index=False)

print("Importing Deliveries (~30-40 seconds)...")
df_deliveries.to_sql("deliveries", con=engine, if_exists="append", index=False, chunksize=5000)

print("All tables successfully loaded into MySQL!")

Importing Matches...


C:\Users\sujit\AppData\Local\Temp\ipykernel_36072\3845641847.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_matches.to_sql("matches", con=engine, if_exists="append", index=False, chunksize=1000)


AttributeError: 'Engine' object has no attribute 'cursor'